In [2]:
# init
import importlib, sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

import superconductivity.api as sc

from superconductivity.api import G0_muS
from superconductivity.api import NDArray64

from IPython import get_ipython

from tqdm import tqdm

Textwidth: float = 4.25279  # in
Textheight: float = 6.85173  # in

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic("config", "InlineBackend.rc = {'figure.dpi': 300}")

In [82]:
# define recification & calibration
from superconductivity.utilities.functions.upsampling import upsample


def get_rectification(y, x, xbins, odd_parity=True):

    ylen = y.shape[0]
    Arect = np.full((ylen), np.nan)

    xbins0 = np.copy(xbins)
    xbins1 = -np.flip(xbins)

    # xup = upsample(z=x, N_up=100)
    # yup = upsample(z=y, N_up=100, axis=1)

    y0 = sc.nanbin_y_over_x(y, x, xbins0, axis=1)
    y1 = sc.nanbin_y_over_x(y, x, xbins1, axis=1)

    if odd_parity:
        y1 *= -1

    yeven = y0 - y1
    yodd = np.abs(y0 + y1)

    nan0, nan1 = np.isnan(y0), np.isnan(y1)
    mask = np.logical_not(np.logical_or(nan0, nan1))
    check = np.sum(mask, axis=1) >= 2

    for i in range(ylen):
        maski = mask[i, :]
        if check[i]:
            Yeven = np.trapezoid(yeven[i, maski], xbins[maski])
            Yodd = np.trapezoid(yodd[i, maski], xbins[maski])
            if Yodd != 0:
                Arect[i] = Yeven / Yodd
    return Arect


def calibrate_T(x, Tbase_K, Tc_K, eta_T, Toff_K):
    return np.clip(eta_T * x + Toff_K, Tbase_K, Tc_K)


def calibrate_A(Aout_mV, etaA):
    return etaA * Aout_mV


# define recification & calibration
from superconductivity.utilities.functions.upsampling import upsample


def get_rectification_map(y, x, xbins, odd_parity=True):

    ylen = y.shape[0]

    xbins0 = np.copy(xbins)
    xbins1 = -np.flip(xbins)

    # xup = upsample(z=x, N_up=100)
    # yup = upsample(z=y, N_up=100, axis=1)

    X, _ = np.meshgrid(x, np.arange(ylen))
    y0 = sc.nanbin_y_over_x(y, X, xbins0, axis=1)
    y1 = np.flip(sc.nanbin_y_over_x(y, X, xbins1, axis=1))

    if odd_parity:
        y1 *= -1

    yeven = y0 - y1
    yodd = np.abs(y0 + y1)

    return np.divide(
        yeven,
        yodd,
        out=np.full_like(yodd, np.nan, dtype=float),
        where=yodd != 0.0,
    )

In [83]:
# start parameter

GN_G0 = 0.09274607247459751
T_K = 0.4424109523100265
Delta_meV = 0.20307510786980312
gamma_meV = 0.005524997349929867
sigmaV_mV = 0.040796080261556025

# Irradadiation ($\nu$, $A$)

In [84]:
# fit zero irradiation data

Aouti_mV = 200
nui_GHz = 7.78

data = np.load(f"irradiation/nu_{nui_GHz:.2f}GHz/eva.npz")

Vbias_mV = data["Vbias_mV"]
Aout_mV = data["Aout_mV"]
Iexp_nA = data["Iexp_nA"]
Tbath_K = data["Tbath_K"]

settings = {
    "GN_G0": (0.09, 0.05, 0.15, False),
    "T_K": (0.3, 0.0, 1.2, False),
    "Delta_meV": (0.188, 0.18, 0.210, False),
    "gamma_meV": (1e-3, 1e-9, 25e-3, False),
    "sigmaV_mV": (0.01, 0.0, 1.0, False),
    "A_mV": (Aouti_mV * 5e-4, 0.0, 4.0, False),
    "nu_GHz": (nui_GHz, 1.0, 20.0, True),
}

# fit only the noise-capable model
from bcs_fit import fit_bcs_conv_noise

i = np.argmin(np.abs(Aout_mV - Aouti_mV))
fit_row, noise_solution = fit_bcs_conv_noise(
    Vbias_mV,
    Iexp_nA[i, :],
    settings,
    maxfev=20_000,
)

Ifit_nA = noise_solution["I_fit_nA"]
fit_values = {parameter.name: parameter.value for parameter in noise_solution["params"]}
fit_errors = {parameter.name: parameter.error for parameter in noise_solution["params"]}

print("GN_G0", fit_values["GN_G0"], fit_errors["GN_G0"])
print("T_K", fit_values["T_K"], fit_errors["T_K"])
print("Delta_meV", fit_values["Delta_meV"], fit_errors["Delta_meV"])
print("gamma_meV", fit_values["gamma_meV"], fit_errors["gamma_meV"])
print("sigmaV_mV", fit_values["sigmaV_mV"], fit_errors["sigmaV_mV"])
print("A_mV", fit_values["A_mV"], fit_errors["A_mV"])
print("nu_GHz", fit_values["nu_GHz"], fit_errors["nu_GHz"])
print("eta", fit_values["A_mV"] / Aouti_mV, fit_errors["A_mV"] / Aouti_mV)
print("Tbath_K", Tbath_K[i])

np.savez(
    "irradiation/singleiv-fit.npz",
    Vbias_mV=Vbias_mV,
    Aout_mV=Aout_mV[i],
    Tbath_K=Tbath_K[i],
    Ifit_nA=Ifit_nA,
    Iexp_nA=Iexp_nA[i, :],
    values=fit_values,
    errors=fit_errors,
)

GN_G0 0.09274607247459751 8.295027302554413e-05
T_K 0.5346641575097719 0.04156769791869541
Delta_meV 0.20307510786980312 0.0010161817927575605
gamma_meV 0.005486734324060615 0.0017795066158114903
sigmaV_mV 0.04098241952449757 0.0037229078476626845
A_mV 0.10589254676874743 0.001226698234848312
nu_GHz 7.78 0.0
eta 0.0005294627338437372 6.1334911742415604e-06
Tbath_K 0.27699132417582417


In [85]:
# # fit_data
# import numpy as np
# from tqdm import tqdm

# from bcs_fit import fit_bcs_conv_noise

# nu_GHz = np.arange(7.75, 8.05, 0.01)

# for i, nui_GHz in enumerate(nu_GHz):
#     # load raw data
#     data = np.load(f"irradiation/nu_{nui_GHz:3.2f}GHz/eva.npz")

#     Vbias_mV = data["Vbias_mV"]
#     Iexp_nA = data["Iexp_nA"]

#     settings = {
#         "GN_G0": (0.09274607247459751, 0.1, 0.3, True),
#         "T_K": (0.2, 0.0, 1.5, False),
#         "Delta_meV": (0.20307510786980312, 0.180, 0.210, True),
#         "gamma_meV": (0.005486734324060615, 1e-9, 25e-3, True),
#         "sigmaV_mV": (0.04098241952449757, 0.0, 4.0, True),
#         "A_mV": (0.001, 0.0, 10.0, False),
#         "nu_GHz": (nui_GHz, 1.0, 20.0, True),
#     }
#     current_settings = dict(settings)
#     n_curves = Iexp_nA.shape[0]

#     Ifit_nA = np.full_like(Iexp_nA, np.nan, dtype=float)
#     fit_rows = []
#     solutions = []

#     fit_values = {name: np.full(n_curves, np.nan) for name in settings}
#     fit_errors = {name: np.full(n_curves, np.nan) for name in settings}

#     for i in tqdm(range(n_curves), desc="BCS fits"):
#         current_I_nA = Iexp_nA[i, :]

#         try:
#             fit_row, solution = fit_bcs_conv_noise(
#                 Vbias_mV,
#                 current_I_nA,
#                 current_settings,
#                 maxfev=2_000,
#             )
#         except (RuntimeError, ValueError, KeyError) as error:
#             print(f"curve {i} failed: {error}")
#             fit_rows.append({"index": i, "ok": False, "error": str(error)})
#             solutions.append(None)
#             continue

#         fit_rows.append({"index": i, **fit_row})
#         solutions.append(solution)
#         Ifit_nA[i, :] = solution["I_fit_nA"]

#         for parameter in solution["params"]:
#             fit_values[parameter.name][i] = parameter.value
#             fit_errors[parameter.name][i] = parameter.error

#         # Warm-start the next curve, keeping bounds and fixed flags unchanged.
#         current_settings = {
#             parameter.name: (
#                 parameter.value,
#                 current_settings[parameter.name][1],
#                 current_settings[parameter.name][2],
#                 current_settings[parameter.name][3],
#             )
#             for parameter in solution["params"]
#         }

#     np.savez_compressed(
#         f"irradiation/nu_{nui_GHz:3.2f}GHz/fit_data4.npz",
#         Ifit_nA=Ifit_nA,
#         **{f"value_{name}": data for name, data in fit_values.items()},
#         **{f"error_{name}": data for name, data in fit_errors.items()},
#     )

In [86]:
# calibrate and calculate rectification
from pathlib import Path
from superconductivity.utilities.functions.upsampling import upsample
from scipy.optimize import curve_fit

nu_GHz = np.arange(7.75, 8.05, 0.01)
Aout_mV = np.arange(0, 810.0, 10)
Vbins = np.linspace(0, 4, 1001)

Acal = np.linspace(0.0, 10.0, 101)

AI = np.full((nu_GHz.shape[0], Aout_mV.shape[0], Vbins.shape[0]), np.nan)
AdG = np.full((nu_GHz.shape[0], Aout_mV.shape[0], Vbins.shape[0]), np.nan)
AdR = np.full((nu_GHz.shape[0], Aout_mV.shape[0], Vbins.shape[0]), np.nan)

eta_Afit_Aout = np.full_like(nu_GHz, np.nan)

eta_Tfit_Aout = np.full_like(nu_GHz, np.nan)
Tbase_Tfit_Aout = np.full_like(nu_GHz, np.nan)
Tc_Tfit_Aout = np.full_like(nu_GHz, np.nan)
Toff_Tfit_Aout = np.full_like(nu_GHz, np.nan)

eta_Tbath_Aout = np.full_like(nu_GHz, np.nan)
Tbase_Tbath_Aout = np.full_like(nu_GHz, np.nan)
Tc_Tbath_Aout = np.full_like(nu_GHz, np.nan)
Toff_Tbath_Aout = np.full_like(nu_GHz, np.nan)

eta_Tfit_Afit = np.full_like(nu_GHz, np.nan)
Tbase_Tfit_Afit = np.full_like(nu_GHz, np.nan)
Tc_Tfit_Afit = np.full_like(nu_GHz, np.nan)
Toff_Tfit_Afit = np.full_like(nu_GHz, np.nan)

eta_Tbath_Afit = np.full_like(nu_GHz, np.nan)
Tbase_Tbath_Afit = np.full_like(nu_GHz, np.nan)
Tc_Tbath_Afit = np.full_like(nu_GHz, np.nan)
Toff_Tbath_Afit = np.full_like(nu_GHz, np.nan)

eta_Tfit_Tbath = np.full_like(nu_GHz, np.nan)
Tbase_Tfit_Tbath = np.full_like(nu_GHz, np.nan)
Tc_Tfit_Tbath = np.full_like(nu_GHz, np.nan)
Toff_Tfit_Tbath = np.full_like(nu_GHz, np.nan)

Afit_Aout_mV = np.zeros((nu_GHz.shape[0], Aout_mV.shape[0]))
eta_Aout = np.zeros((nu_GHz.shape[0], Aout_mV.shape[0]))
Tfit_Aout_K = np.zeros((nu_GHz.shape[0], Aout_mV.shape[0]))
Tbath_Aout_K = np.zeros((nu_GHz.shape[0], Aout_mV.shape[0]))
AI_Aout = np.zeros((nu_GHz.shape[0], Aout_mV.shape[0]))
AdG_Aout = np.zeros((nu_GHz.shape[0], Aout_mV.shape[0]))
AdR_Aout = np.zeros((nu_GHz.shape[0], Aout_mV.shape[0]))

Afit_Acal_mV = np.zeros((nu_GHz.shape[0], Acal.shape[0]))
eta_Acal = np.zeros((nu_GHz.shape[0], Acal.shape[0]))
Tfit_Acal_K = np.zeros((nu_GHz.shape[0], Acal.shape[0]))
Tbath_Acal_K = np.zeros((nu_GHz.shape[0], Acal.shape[0]))
AI_Acal = np.zeros((nu_GHz.shape[0], Acal.shape[0]))
AdG_Acal = np.zeros((nu_GHz.shape[0], Acal.shape[0]))
AdR_Acal = np.zeros((nu_GHz.shape[0], Acal.shape[0]))

for i, nui_GHz in enumerate(tqdm(nu_GHz)):
    if True:
        datafit = np.load(f"irradiation/nu_{nui_GHz:3.2f}GHz/fit_data4.npz")
        dataexp = np.load(f"irradiation/nu_{nui_GHz:3.2f}GHz/eva.npz")

        Afit_Aout_mV[i, :] = datafit["value_A_mV"]
        Tfit_Aout_K[i, :] = datafit["value_T_K"]
        Tbath_Aout_K[i, :] = dataexp["Tbath_K"]

        # Aout_mV = dataexp["Aout_mV"]
        Vbias_mV = dataexp["Vbias_mV"]
        Ibias_nA = dataexp["Ibias_nA"]

        dGexp_G0 = dataexp["dGexp_G0"]
        dRexp_R0 = dataexp["dRexp_R0"]
        Iexp_nA = dataexp["Iexp_nA"]

        Vbias = Vbias_mV / Delta_meV
        Ibias = Ibias_nA / (Delta_meV * GN_G0 * sc.G0_muS)

        Iexp = Iexp_nA / (Delta_meV * GN_G0 * sc.G0_muS)
        dGexp = dGexp_G0 / (GN_G0)
        dRexp = dRexp_R0 * (GN_G0)

        AI[i, :, :] = get_rectification_map(Iexp, Vbias, Vbins, odd_parity=True)
        AdG[i, :, :] = get_rectification_map(dGexp, Vbias, Vbins, odd_parity=False)
        AdR[i, :, :] = get_rectification_map(dRexp, Vbias, Vbins, odd_parity=False)

        AI_Aout[i, :] = get_rectification(Iexp, Vbias, Vbins, odd_parity=True)
        AdG_Aout[i, :] = get_rectification(dGexp, Vbias, Vbins, odd_parity=False)
        AdR_Aout[i, :] = get_rectification(dRexp, Vbias, Vbins, odd_parity=False)

        eta_Aout[i, :] = np.divide(
            Afit_Aout_mV[i, :],
            Aout_mV,
            out=np.full_like(Afit_Aout_mV[i, :], 0.0, dtype=float),
            where=Aout_mV != 0.0,
        )

        eta_Afit_Aout[i] = np.median(eta_Aout[i, :])
        Afit = eta_Afit_Aout[i] * upsample(Aout_mV) / (sc.h_pVs * nui_GHz)

        Ical = sc.bin(upsample(Iexp, axis=0), Afit, Acal, axis=0)
        dGcal = sc.bin(upsample(dGexp, axis=0), Afit, Acal, axis=0)
        dRcal = sc.bin(upsample(dRexp, axis=0), Afit, Acal, axis=0)

        AI_Acal[i, :] = sc.bin(upsample(AI_Aout[i, :]), Afit, Acal, axis=0)
        AdG_Acal[i, :] = sc.bin(upsample(AdG_Aout[i, :]), Afit, Acal, axis=0)
        AdR_Acal[i, :] = sc.bin(upsample(AdR_Aout[i, :]), Afit, Acal, axis=0)

        Afit_Acal_mV[i, :] = sc.bin(upsample(Afit_Aout_mV[i, :]), Afit, Acal)
        Tfit_Acal_K[i, :] = sc.bin(upsample(Tfit_Aout_K[i, :]), Afit, Acal)
        eta_Acal[i, :] = sc.bin(upsample(eta_Aout[i, :]), Afit, Acal)
        Tbath_Acal_K[i, :] = sc.bin(upsample(Tbath_Aout_K[i, :]), Afit, Acal)

        out_path = Path(f"irradiation/nu_{nui_GHz:3.2f}GHz/cal.npz")
        out_path.parent.mkdir(parents=True, exist_ok=True)

        np.savez_compressed(
            out_path,
            Vbias=Vbias,
            Ibias=Ibias,
            Acal=Acal,
            Ical=Ical,
            dGcal=dGcal,
            dRcal=dRcal,
            Tbath_K=Tbath_Acal_K[i, :],
            nu_GHz=nui_GHz,
            Delta_meV=Delta_meV,
            GN_G0=GN_G0,
        )

        popt, _ = curve_fit(
            calibrate_T,
            Aout_mV,
            Tfit_Aout_K[i, :],
            p0=[0.2, 1.2, 1e-3, 0.0],
            bounds=([0.0, 1.0, 0.0, -1.0], [1.0, 1.5, 1.0, 1.0]),
        )
        Tbase_Tfit_Aout[i], Tc_Tfit_Aout[i], eta_Tfit_Aout[i], Toff_Tfit_Aout[i] = popt

        popt, _ = curve_fit(
            calibrate_T,
            Aout_mV,
            Tbath_Aout_K[i, :],
            p0=[0.3, 1.2, 1e-3, 0.0],
            bounds=([0.0, 0.0, 0.0, -1.0], [1.0, 1.5, 1.0, 1.0]),
        )
        Tbase_Tbath_Aout[i], Tc_Tbath_Aout[i], eta_Tbath_Aout[i], Toff_Tbath_Aout[i] = (
            popt
        )

        popt, _ = curve_fit(
            calibrate_T,
            Afit_Aout_mV[i, :],
            Tfit_Aout_K[i, :],
            p0=[0.3, 1.2, 2.3, 0.0],
            bounds=([0.0, 1.0, 0.0, -1.0], [1.0, 1.5, 10.0, 1.0]),
        )
        Tbase_Tfit_Afit[i], Tc_Tfit_Afit[i], eta_Tfit_Afit[i], Toff_Tfit_Afit[i] = popt

        popt, _ = curve_fit(
            calibrate_T,
            Afit_Aout_mV[i, :],
            Tbath_Aout_K[i, :],
            p0=[0.3, 1.2, 2.3, 0.0],
            bounds=([0.0, 0.0, 0.0, -1.0], [1.0, 1.5, 10.0, 1.0]),
        )
        Tbase_Tbath_Afit[i], Tc_Tbath_Afit[i], eta_Tbath_Afit[i], Toff_Tbath_Afit[i] = (
            popt
        )

        popt, _ = curve_fit(
            calibrate_T,
            Tbath_Aout_K[i, :],
            Tfit_Aout_K[i, :],
            p0=[0.3, 1.2, 2.3, 0.0],
            bounds=([0.0, 0.0, 0.0, -1.0], [1.0, 1.5, 10.0, 1.0]),
        )
        Tbase_Tfit_Tbath[i], Tc_Tfit_Tbath[i], eta_Tfit_Tbath[i], Toff_Tfit_Tbath[i] = (
            popt
        )

np.savez_compressed(
    "irradiation/fit.npz",
    nu_GHz=nu_GHz,
    eta_Afit_Aout=eta_Afit_Aout,
    eta_Tfit_Aout=eta_Tfit_Aout,
    Tbase_Tfit_Aout=Tbase_Tfit_Aout,
    Tc_Tfit_Aout=Tc_Tfit_Aout,
    Toff_Tfit_Aout=Toff_Tfit_Aout,
    eta_Tbath_Aout=eta_Tbath_Aout,
    Tbase_Tbath_Aout=Tbase_Tbath_Aout,
    Tc_Tbath_Aout=Tc_Tbath_Aout,
    Toff_Tbath_Aout=Toff_Tbath_Aout,
    eta_Tfit_Afit=eta_Tfit_Afit,
    Tbase_Tfit_Afit=Tbase_Tfit_Afit,
    Tc_Tfit_Afit=Tc_Tfit_Afit,
    Toff_Tfit_Afit=Toff_Tfit_Afit,
    eta_Tbath_Afit=eta_Tbath_Afit,
    Tbase_Tbath_Afit=Tbase_Tbath_Afit,
    Tc_Tbath_Afit=Tc_Tbath_Afit,
    Toff_Tbath_Afit=Toff_Tbath_Afit,
    eta_Tfit_Tbath=eta_Tfit_Tbath,
    Tbase_Tfit_Tbath=Tbase_Tfit_Tbath,
    Tc_Tfit_Tbath=Tc_Tfit_Tbath,
    Toff_Tfit_Tbath=Toff_Tfit_Tbath,
    Aout_mV=Aout_mV,
    Afit_Aout_mV=Afit_Aout_mV,
    eta_Aout=eta_Aout,
    Tfit_Aout_K=Tfit_Aout_K,
    Tbath_Aout_K=Tbath_Aout_K,
    AI_Aout=AI_Aout,
    AdG_Aout=AdG_Aout,
    AdR_Aout=AdR_Aout,
    Acal=Acal,
    Afit_Acal_mV=Afit_Acal_mV,
    eta_Acal=eta_Acal,
    Tfit_Acal_K=Tfit_Acal_K,
    Tbath_Acal_K=Tbath_Acal_K,
    AI_Acal=AI_Acal,
    AdG_Acal=AdG_Acal,
    AdR_Acal=AdR_Acal,
    AI=AI,
    AdG=AdG,
    AdR=AdR,
)

100%|██████████| 31/31 [01:16<00:00,  2.47s/it]


In [88]:
# load calibration data
datafit = np.load(f"irradiation/fit.npz")
nu_GHz = datafit["nu_GHz"]
Vbins = datafit["Vbins"]
Acal = datafit["Acal"]
eta_fit = datafit["eta_fit"]
eta_cal = datafit["eta_cal"]
ArectIexp = datafit["ArectIexp"]
ArectIcal = datafit["ArectIcal"]
ArectdGexp = datafit["ArectdGexp"]
ArectdGcal = datafit["ArectdGcal"]
ArectdRexp = datafit["ArectdRexp"]
ArectdRcal = datafit["ArectdRcal"]
Delta_meV = datafit["Delta_meV"]
GN_G0 = datafit["GN_G0"]

ii = np.array([0, 12, 14, 16, 21, 30])
for i, nui_GHz in enumerate(nu_GHz):
    datacal = np.load(f"irradiation/nu_{nui_GHz:3.2f}GHz/cal.npz")
    Vbias = datacal["Vbias"]
    Ibias = datacal["Ibias"]
    Acal = datacal["Acal"]
    Ical = datacal["Ical"]
    dGcal = datacal["dGcal"]
    dRcal = datacal["dRcal"]
    if i in ii:
        pass

KeyError: 'Vbins is not a file in the archive'

In [69]:
# show recification
plt.close("all")

dnu_GHz = (nu_GHz[1] - nu_GHz[0]) / 2
nulim = nu_GHz[0] - dnu_GHz, nu_GHz[-1] + dnu_GHz

dVbins = (Vbins[1] - Vbins[0]) / 2
Vlim = Vbins[0] - dVbins, Vbins[-1] + dVbins

dAcal = (Acal[1] - Acal[0]) / 2
Alim = Acal[0] - dAcal, Acal[-1] + dAcal

plt.imshow(
    ArectIexp,
    aspect="auto",
    extent=(
        *Alim,
        *nulim,
    ),
    origin="lower",
    # clim=(0, 2),
)
plt.xlabel("$eA/h\\nu$")
plt.ylabel("$\\nu$ (GHz)")
plt.show()

In [70]:
# show calibration
plt.close("all")
nu_GHz = np.arange(7.75, 8.05, 0.01)
ii = np.array([0, 12, 14, 16, 21, 30])
for i, nui_GHz in enumerate(nu_GHz):
    # if i in ii:
    if True:
        datacal = np.load(f"irradiation/nu_{nui_GHz:3.2f}GHz/cal.npz")
        Vbias = datacal["Vbias"]
        Ibias = datacal["Ibias"]
        Acal = datacal["Acal"]
        Ical = datacal["Ical"]
        dGcal = datacal["dGcal"]
        dRcal = datacal["dRcal"]

        dAcal = (Acal[1] - Acal[0]) / 2
        dVbias = (Vbias[1] - Vbias[0]) / 2
        Alim = Acal[0] - dAcal, Acal[-1] + dAcal
        Vlim = Vbias[0] - dVbias, Vbias[-1] + dVbias
        slope = (sc.h_pVs * nui_GHz) / Delta_meV

        plt.figure(i)
        plt.imshow(
            dGcal,
            aspect="auto",
            extent=(*Vlim, *Alim),
            origin="lower",
            clim=(0, 2),
        )
        plt.plot(+2 + Acal * slope, Acal, color=sc.seeblau100)
        plt.plot(+2 - Acal * slope, Acal, color=sc.seeblau100)
        plt.plot(-2 + Acal * slope, Acal, color=sc.seeblau100)
        plt.plot(-2 - Acal * slope, Acal, color=sc.seeblau100)
        plt.xlabel("$eV/\\Delta$")
        plt.xlabel("$eA/h\\nu$")
        plt.savefig(f"irradiation/cal/cal_{i}_{nui_GHz:.2f}GHz.png")

/var/folders/kc/8fnzl3f94vxgl8w4wm3wfvk80000gn/T/ipykernel_12414/2292286709.py:22: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(i)


# Frequency Studies

In [24]:
# load raw data
paths = [
    "nu0_0.5V",
    "nu1_0.5V",
    "nu2_0.5V",
    "nu3_0.4V",
    "nu4_0.4V",
    "nu5_0.1V",
    "nu6_0.1V",
    "nu7_0.1V",
]
for path in paths:
    dataeva = np.load(f"{path}/eva.npz")

    Vbias_mV = dataeva["Vbias_mV"]
    Ibias_nA = dataeva["Ibias_nA"]

    nu_GHz = dataeva["nu_GHz"]
    Aout_mV = dataeva["Aout_mV"]

    Iexp_nA = dataeva["Iexp_nA"]
    dGexp_G0 = dataeva["dGexp_G0"]
    dRexp_R0 = dataeva["dRexp_R0"]
    Tbath_K = dataeva["Tbath_K"]

    nu_GHz[0] = 1e-9

    eta = np.full_like(nu_GHz, np.nan)
    Afit_mV = np.full_like(nu_GHz, np.nan)
    Aerr_mV = np.full_like(nu_GHz, np.nan)
    Ifit_mV = np.full_like(Iexp_nA, np.nan)

    Vbias = Vbias_mV / Delta_meV
    Ibias = Ibias_nA / (GN_G0 * sc.G0_muS * Delta_meV)
    Iexp = Iexp_nA / (GN_G0 * sc.G0_muS * Delta_meV)
    dGexp = dGexp_G0 / GN_G0
    dRexp = dRexp_R0 * GN_G0

    Vbins = np.linspace(0, 4, 801)
    ArectI = get_rectification(Iexp, Vbias, Vbins, odd_parity=True)
    ArectdG = get_rectification(dGexp, Vbias, Vbins, odd_parity=False)
    ArectdR = get_rectification(dRexp, Ibias, Vbins, odd_parity=False)

    from bcs_fit import fit_bcs_conv_noise

    for i, nui_GHz in enumerate(tqdm(nu_GHz)):
        settings = {
            "GN_G0": (GN_G0, 0.1, 0.3, True),
            "T_K": (T_K, 0.0, 1.2, True),
            "Delta_meV": (Delta_meV, 0.180, 0.210, True),
            "gamma_meV": (gamma_meV, 1e-9, 25e-3, True),
            "sigmaV_mV": (sigmaV_mV, 0.0, 1.0, True),
            "A_mV": (0.001, 0.0, 4.0, False),
            "nu_GHz": (nui_GHz, 0.0, 20.0, True),
        }
        mask = np.isfinite(Iexp_nA[i, :])
        fit_row, noise_solution = fit_bcs_conv_noise(
            Vbias_mV[mask],
            Iexp_nA[i, mask],
            settings,
            maxfev=20_000,
        )
        Ifit_mV[i, mask] = noise_solution["I_fit_nA"]
        Afit_mV[i] = noise_solution["params"][4].value
        Aerr_mV[i] = noise_solution["params"][4].error
        eta[i] = Afit_mV[i] / Aout_mV

    eta[0] = np.nan
    np.savez_compressed(
        f"{path}/fit.npz",
        Vbias=Vbias,
        Ibias=Ibias,
        Iexp=Iexp,
        dGexp=dGexp,
        dRexp=dRexp,
        Tbath_K=Tbath_K,
        Ifit_mV=Ifit_mV,
        Afit_mV=Afit_mV,
        Aerr_mV=Aerr_mV,
        eta=eta,
        Vbins=Vbins,
        ArectI=ArectI,
        ArectdG=ArectdG,
        ArectdR=ArectdR,
        Aout_mV=Aout_mV,
        nu_GHz=nu_GHz,
        Delta_meV=Delta_meV,
        GN_G0=GN_G0,
    )

100%|██████████| 192/192 [01:08<00:00,  2.79it/s]


In [15]:
# show eta, ArectIexp, Tbath over nu

fitdata = np.load("irradiation/fit.npz")
A0_mV = 500
A1_mV = 400
A2_mV = 100
ia0 = np.argmin(np.abs(fitdata["Aout_mV"] - A0_mV))
ia1 = np.argmin(np.abs(fitdata["Aout_mV"] - A1_mV))
ia2 = np.argmin(np.abs(fitdata["Aout_mV"] - A2_mV))

paths = [
    "nu0_0.5V",
    "nu1_0.5V",
    "nu2_0.5V",
    "nu3_0.4V",
    "nu4_0.4V",
    "nu5_0.1V",
    "nu6_0.1V",
    "nu7_0.1V",
]
antenna = [True, True, True, True, False, True, False, True]


if _ip is not None:
    _ip.run_line_magic("matplotlib", "qt")
plt.close("all")


for i, path in enumerate(paths):
    tempfit = np.load(f"{path}/fit.npz")
    tempeva = np.load(f"{path}/eva.npz")
    plt.figure(0)
    plt.plot(
        tempfit["nu_GHz"],
        tempfit["eta"],
        ".-",
        ms=5,
        lw=1,
        label=f"{path} {"antenna" if antenna[i] else "stripline"}",
    )
    plt.figure(1)
    plt.plot(
        tempfit["nu_GHz"],
        tempfit["ArectI"],
        ".-",
        ms=5,
        lw=1,
        label=f"{path} {"antenna" if antenna[i] else "stripline"}",
    )
    plt.figure(2)
    plt.plot(
        tempfit["nu_GHz"],
        tempfit["Tbath_K"],
        ".-",
        ms=5,
        lw=1,
        label=f"{path} {"antenna" if antenna[i] else "stripline"}",
    )

plt.figure(0)
plt.plot(
    fitdata["nu_GHz"],
    fitdata["eta_fit"][:, ia0],
    ".-",
    ms=5,
    lw=1,
    label=f"irra_{A0_mV/1000:0.1f}V_antenna",
)
plt.plot(
    fitdata["nu_GHz"],
    fitdata["eta_fit"][:, ia1],
    ".-",
    ms=5,
    lw=1,
    label=f"irra_{A1_mV/1000:0.1f}V_antenna",
)
plt.plot(
    fitdata["nu_GHz"],
    fitdata["eta_fit"][:, ia2],
    ".-",
    ms=5,
    lw=1,
    label=f"irra_{A2_mV/1000:0.1f}V_antenna",
)
plt.legend()
plt.grid()
plt.xlabel("$\\nu$ (GHz)")
plt.ylabel("$\\eta$")
plt.xlim(7.7, 8.3)

plt.figure(1)
plt.plot(
    fitdata["nu_GHz"],
    fitdata["ArectIexp"][:, ia0],
    ".-",
    ms=5,
    lw=1,
    label=f"irra_{A0_mV/1000:0.1f}V_antenna",
)
plt.plot(
    fitdata["nu_GHz"],
    fitdata["ArectIexp"][:, ia1],
    ".-",
    ms=5,
    lw=1,
    label=f"irra_{A1_mV/1000:0.1f}V_antenna",
)
plt.plot(
    fitdata["nu_GHz"],
    fitdata["ArectIexp"][:, ia2],
    ".-",
    ms=5,
    lw=1,
    label=f"irra_{A2_mV/1000:0.1f}V_antenna",
)
plt.legend()
plt.grid()
plt.xlabel("$\\nu$ (GHz)")
plt.ylabel("$A_\\mathrm{rect}(I)$")
plt.xlim(7.7, 8.3)

plt.figure(2)
plt.plot(
    fitdata["nu_GHz"],
    fitdata["Tbathexp_K"][:, ia0],
    ".-",
    ms=5,
    lw=1,
    label=f"irra_{A0_mV/1000:0.1f}V_antenna",
)
plt.plot(
    fitdata["nu_GHz"],
    fitdata["Tbathexp_K"][:, ia1],
    ".-",
    ms=5,
    lw=1,
    label=f"irra_{A1_mV/1000:0.1f}V_antenna",
)
plt.plot(
    fitdata["nu_GHz"],
    fitdata["Tbathexp_K"][:, ia2],
    ".-",
    ms=5,
    lw=1,
    label=f"irra_{A2_mV/1000:0.1f}V_antenna",
)
plt.legend()
plt.grid()
plt.xlabel("$\\nu$ (GHz)")
plt.ylabel("$T_\\mathrm{bath}$ (K)")
plt.xlim(7.7, 8.3)

(7.7, 8.3)